In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["numpy", "Pillow"]
ASSETS = [{"name": "img_test.png", "url": "/labs/3.2.3/img_test.png", "size": 17697, "sha256": "7e688ab62e96522f26d84cc0178621794c5f2f705e6a323ded11f3f668a6e124"}]

if PACKAGES:
    await piplite.install(PACKAGES)

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    response = await pyfetch(asset["url"])
    payload = await response.bytes()
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 3.2.3 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
# JupyterLite 浏览器 ONNX Runtime：从现有分片清单加载、校验、缓存并执行模型
from pyodide.code import run_js

_browser_ort_run = run_js(r"""
async (taskCode) => {
  const origin = self.location.origin;
  const digest = async buffer => {
    const hash = await crypto.subtle.digest("SHA-256", buffer);
    return [...new Uint8Array(hash)].map(value => value.toString(16).padStart(2, "0")).join("");
  };
  const response = await fetch(`${origin}/labs/manifest.json`);
  if (!response.ok) throw new Error("模型清单加载失败");
  const manifest = await response.json();
  const model = manifest.tasks?.[taskCode]?.model;
  if (!model) throw new Error(`任务 ${taskCode} 没有浏览器模型`);

  const cache = await caches.open("ai-trainer-onnx-v2");
  const cacheKey = new Request(`${origin}/labs/model-cache/${model.modelId}.onnx`);
  let modelBytes;
  const cached = await cache.match(cacheKey);
  if (cached) {
    const cachedBuffer = await cached.arrayBuffer();
    if (await digest(cachedBuffer) === model.sha256) modelBytes = new Uint8Array(cachedBuffer);
  }
  if (!modelBytes) {
    const chunks = [];
    let total = 0;
    for (const part of model.chunks) {
      const partResponse = await fetch(`${origin}${part.url}`);
      if (!partResponse.ok) throw new Error(`模型分片加载失败：${part.url}`);
      const partBuffer = await partResponse.arrayBuffer();
      if (await digest(partBuffer) !== part.sha256) throw new Error(`模型分片校验失败：${part.url}`);
      const bytes = new Uint8Array(partBuffer);
      chunks.push(bytes);
      total += bytes.byteLength;
    }
    modelBytes = new Uint8Array(total);
    let offset = 0;
    for (const chunk of chunks) { modelBytes.set(chunk, offset); offset += chunk.byteLength; }
    if (await digest(modelBytes.buffer) !== model.sha256) throw new Error("模型完整性校验失败");
    await cache.put(cacheKey, new Response(modelBytes, { headers: { "content-type": "application/octet-stream" } }));
  }

  const runtimeBase = `${origin}/labs/runtime/onnxruntime-web/1.27.0/`;
  const ort = await import(`${runtimeBase}ort.wasm.bundle.min.js`);
  ort.env.wasm.numThreads = 1;
  ort.env.wasm.wasmPaths = runtimeBase;
  const session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
  const feeds = {};
  const inputs = [];
  for (const [index, name] of session.inputNames.entries()) {
    const meta = session.inputMetadata[index];
    if (!meta?.isTensor) continue;
    const shape = (meta.shape?.length ? meta.shape : [1]).map(value => Number.isInteger(value) && value > 0 ? value : 1);
    const size = shape.reduce((left, right) => left * right, 1);
    const sourceType = String(meta.type ?? "float32");
    const type = sourceType.includes("uint8") ? "uint8" : sourceType.includes("int32") ? "int32" : "float32";
    const data = type === "uint8" ? new Uint8Array(size) : type === "int32" ? new Int32Array(size) : new Float32Array(size);
    feeds[name] = new ort.Tensor(type, data, shape);
    inputs.push({ name, type, shape });
  }
  const outputMap = await session.run(feeds);
  const outputs = Object.entries(outputMap).map(([name, tensor]) => ({ name, type: tensor.type, shape: [...tensor.dims] }));
  return { model: model.name, runtime: model.browserRuntime, variant: model.sourceVariant, inputs, outputs };
}
""")

async def browser_onnx_check(task_code="3.2.3"):
    """执行一次真实的浏览器 WASM 推理，并返回张量结构。"""
    result = await _browser_ort_run(task_code)
    return result.to_py() if hasattr(result, "to_py") else result

onnx_result = await browser_onnx_check()
onnx_result


In [ ]:
# 导入必要的库
import numpy as np
from PIL import Image
import onnxruntime as ort


# 定义预处理函数，用于将图片转换为模型所需的输入格式
def preprocess(image_path):
    input_shape = (1, 1, 64, 64)    # 模型输入期望的形状，这里是 (N, C, H, W)，N=batch size, C=channels, H=height, W=width
    img = Image.open(image_path).convert('L')    # 打开图像文件并将其转换为灰度图  1分
    img = img.resize((64, 64), Image.ANTIALIAS)    # 调整图像大小到模型输入所需的尺寸
    img_data = np.array(img, dtype=np.float32)    # 将PIL图像对象转换为numpy数组，并确保数据类型是float32
    # 调整数组的形状以匹配模型输入的形状
    img_data = np.expand_dims(img_data, axis=0)  # 添加 batch 维度
    img_data = np.expand_dims(img_data, axis=1)  # 添加 channel 维度
    assert img_data.shape == input_shape, f"Expected shape {input_shape}, but got {img_data.shape}"    # 确保最终的形状与模型输入要求的形状一致
    return img_data    # 返回预处理后的图像数据


# 定义情感类别与数字标签的映射表 3分
emotion_table = {____________}


# 加载模型 3分
ort_session = ____________    # 使用onnxruntime创建一个会话，用于加载并运行模型


# 加载本地图片并进行预处理 3分
input_data = ____________


# 准备输入数据，确保其符合模型输入的要求
ort_inputs = {ort_session.get_inputs()[0].name: input_data}    # ort_session.get_inputs()[0].name 是获取模型的第一个输入的名字


# 运行模型，进行预测 3分
ort_outs = ____________(None, ____________)


# 解码模型输出，找到预测概率最高的情感类别 3分
predicted_label = ____________(ort_outs[0])


# 根据预测的标签找到对应的情感名称 3分
predicted_emotion = ____________[predicted_label]


# 输出预测的情感
print(f"Predicted emotion: {predicted_emotion}")
